### Guided tour of the `model_evaluation` public API on the P2P running example.

## 1. Load models

Use the public `load_bpmn` helper — it dispatches on file suffix (`.xml` / `.bpmn` → XML loader, everything else → Signavio JSON).

In [1]:
from model_evaluation import load_bpmn

path_model1 = "../examples/p2p_running_example.bpmn"
path_model2 = "../examples/p2p_running_example_variant.bpmn"

model_1_json = load_bpmn(path_model1)
model_2_json = load_bpmn(path_model2)

## 2. Element statistics

Quick element-count summary for each model — activities, events, gateways, flows, pools, and lanes.

In [2]:
def count_elements(model):
    """Count BPMN elements in a model."""
    return {
        "activities": len(model.get("activities", [])),
        "events": len(model.get("events", [])),
        "gateways": len(model.get("gateways", [])),
        "sequence_flows": len(model.get("sequenceFlows", [])),
        "message_flows": len(model.get("messageFlows", [])),
        "pools": len(model.get("pools", [])),
        "lanes": sum(len(p.get("lanes", [])) for p in model.get("pools", [])),
    }


for label, model in [("Model 1", model_1_json), ("Model 2", model_2_json)]:
    counts = count_elements(model)
    print(f"{label}: {sum(counts.values())} total elements")
    for key, val in counts.items():
        if val > 0:
            print(f"  • {key.replace('_', ' ').title()}: {val}")
    print()

Model 1: 48 total elements
  • Activities: 14
  • Events: 7
  • Gateways: 7
  • Sequence Flows: 16
  • Pools: 1
  • Lanes: 3

Model 2: 39 total elements
  • Activities: 13
  • Events: 5
  • Gateways: 5
  • Sequence Flows: 13
  • Pools: 1
  • Lanes: 2



## 3. Semantic name normalization

Align element names between the two models using a sentence-transformer-based string similarity (cosine), so semantically equivalent labels get unified before comparison. The threshold controls how aggressive the alignment is.

In [3]:
from model_evaluation import normalize_atomic_names
from model_evaluation.utils import cosine_sim_optimized

threshold = 0.6
print(f"Aligning element names (threshold={threshold})...")

model2_aligned, mappings = normalize_atomic_names(
    model_1_json, model_2_json, cosine_sim_optimized, threshold=threshold
)

total_mappings = sum(len(v) for v in mappings.values())
if total_mappings > 0:
    print(f"✓ Applied {total_mappings} semantic name mappings")
    for elem_type, mapping in mappings.items():
        if mapping:
            first_old, first_new = next(iter(mapping.items()))
            print(f"  • {elem_type}: {len(mapping)} mappings "
                  f"(e.g. '{first_old}' → '{first_new}')")
else:
    print("✓ No mappings needed (names already aligned)")


/Users/I762870/dev/process-evaluation-framework/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Aligning element names (threshold=0.6)...


✓ Applied 18 semantic name mappings
  • activity_names: 7 mappings (e.g. 'Release  payment' → 'Release  payment')
  • event_names: 2 mappings (e.g. 'Need Identified' → 'Need Identified')
  • pool_names: 1 mappings (e.g. 'SAMPLE Tech ORG' → 'SAMPLE ORG')
  • lane_names: 2 mappings (e.g. 'Finance & Procurement' → 'Purchasing')
  • activity_names__subprocess: 3 mappings (e.g. 'Auto-approval' → 'Auto-approval')
  • event_names__subprocess: 2 mappings (e.g. 'Request  denied' → 'Request denied')
  • gateway_names__subprocess: 1 mappings (e.g. 'amount' → 'amount > 5000€')


## 4. Trace extraction

Each model is converted to a Petri net and explored to enumerate the execution traces. Returns sound variants plus partial traces (deadlocks / timeouts), bounded by `timeout_seconds` and `max_loop_depth`.

In [4]:
from model_evaluation import extract_traces

tr1 = extract_traces(model_1_json, timeout_seconds=5, max_loop_depth=3)
tr2 = extract_traces(model2_aligned, timeout_seconds=5, max_loop_depth=3)

print(f"Model 1: {len(tr1.variants)} sound + {len(tr1.partial_traces)} partial traces")
print(f"Model 2: {len(tr2.variants)} sound + {len(tr2.partial_traces)} partial traces")

print("\nFirst 3 traces from Model 1:")
for t in tr1.all_traces()[:3]:
    print(" ", t)
print("\nFirst 3 traces from Model 2:")
for t in tr2.all_traces()[:3]:
    print(" ", t)


Trace extraction recovered partial results for net '<unnamed>': 9 sound variant(s), 2 partial trace(s); 2 loop-cap hit(s)


Trace extraction recovered partial results for net '<unnamed>': 8 sound variant(s), 1 partial trace(s); 1 loop-cap hit(s)


Model 1: 9 sound + 2 partial traces
Model 2: 8 sound + 1 partial traces

First 3 traces from Model 1:
  ['StartNoneEvent', 'Check for approval type', 'Management review', 'Request approved']
  ['StartNoneEvent', 'Check for approval type', 'Management review', 'Request denied']
  ['StartNoneEvent', 'Check for approval type', 'Auto-approval', 'Request approved']

First 3 traces from Model 2:
  ['StartNoneEvent', 'Check for approval type', 'Management review', 'Request approved']
  ['StartNoneEvent', 'Check for approval type', 'Management review', 'Request denied']
  ['Need Identified', 'Create purchase requisition', 'Approval', 'Create purchase order', 'Receive invoice from supplier', 'Confirm tool access provisioned', 'Invoice Verification', 'Perform 2-way match', 'Clarify with supplier', 'Invoice Verification', 'Perform 2-way match', 'Clarify with supplier', 'Invoice Verification', 'Perform 2-way match', 'Release  payment', 'Payment completed']


## 5. Structural / behavioral / hybrid similarity

- **Structural**: weighted score across element / flow / organizational / subprocess categories.
- **Behavioral**: how much the trace sets overlap (set-based metric over whole traces).
- **Hybrid**: a weighted combination of the two.

In [5]:
from model_evaluation import (
    calculate_bpmn_similarity,
    calculate_trace_similarity,
    calculate_hybrid_similarity,
)

struct = calculate_bpmn_similarity(model_1_json, model2_aligned, method="jaccard")
beh = calculate_trace_similarity(tr1, tr2, method="jaccard")
hybrid = calculate_hybrid_similarity(struct, beh, structural_weight=0.5)
print(hybrid)


{'structural': 0.49661064425770307, 'behavioral': 0.17647058823529413, 'structural_weight': 0.5, 'behavioral_weight': 0.5, 'hybrid': 0.3365406162464986}


## 6. N-gram trace comparison

N-grams are length-`n` contiguous activity sequences extracted from each trace. Each trace is wrapped in `<START>`/`<END>` tokens so trace boundaries become distinct n-grams.

- `n=1` → unigrams (multiset of activities)
- `n=2` → bigrams / directly-follows pairs
- larger `n` → longer behavioral motifs

Set-based similarity (jaccard / dice / overlap) on the n-gram sets gives a graded view of behavioral agreement that isn't as harsh as exact-variant matching.

In [6]:
from model_evaluation import calculate_ngram_similarity, extract_ngrams

for n in (1, 2, 3):
    print(f"\nn={n}")
    for method in ("jaccard", "dice", "overlap"):
        score = calculate_ngram_similarity(tr1, tr2, n=n, method=method)
        print(f"  {method:8s}: {score:.2%}")
    s1 = set(extract_ngrams(tr1, n=n))
    s2 = set(extract_ngrams(tr2, n=n))
    print(f"  shared: {len(s1 & s2)}  |  only in M1: {len(s1 - s2)}  |  only in M2: {len(s2 - s1)}")


n=1
  jaccard : 66.67%
  dice    : 80.00%
  overlap : 80.00%
  shared: 16  |  only in M1: 4  |  only in M2: 4

n=2
  jaccard : 43.24%
  dice    : 60.38%
  overlap : 61.54%
  shared: 16  |  only in M1: 11  |  only in M2: 10

n=3
  jaccard : 30.23%
  dice    : 46.43%
  overlap : 46.43%
  shared: 13  |  only in M1: 15  |  only in M2: 15


## Interactive dashboard

The interactive dashboard lives in [`notebooks/dashboard.py`](dashboard.py) as a marimo notebook. It bundles the structural, behavioral (with the n-gram subpanel), and hybrid sections into one reactive view.

Launch it from the repo root with:

```
poetry run marimo edit notebooks/dashboard.py
```